# VascuQuest Parameterized Cohort Manual Qualification

This notebook is the checkpointed real-PWDB qualification route for PR #20.

Run in order:

1. **Setup** — mount Drive, clone the exact PR branch, install VascuQuest.
2. **Source staging** — check only the known `PWDB_3275625` directory; never scan all of MyDrive.
3. **Phase A** — 1 subject × 4 diseases.
4. **Phase B** — 3 subjects × 4 diseases, only after Phase A passes.
5. **Finalize** — write the machine-readable qualification report.

Every completed subject is persisted and verified. PASS remains `MODELLED`; it is not clinical validation.


## 1 — Setup

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os, shutil, subprocess, sys, json

REPO_URL = "https://github.com/KNOWDYN/VascuQuest.git"
QUALIFICATION_REF = "release/parameterized-cohort-qualification"

OUTPUT_BASE = Path("/content/drive/MyDrive/VascuQuest/parameterized_cohort_qualification")
LOCAL_REPO = Path("/content/VascuQuest-qualification")
LOCAL_SOURCE = Path("/content/vascuquest-pwdb-source")
LOCAL_XDG = Path("/content/vascuquest-xdg")

# Known exact source location from the existing VascuQuest/PWDB work.
# A second exact candidate covers the case where the same hierarchy is mounted
# as a Shared Drive. There is deliberately NO recursive Drive search.
DRIVE_PWDB_CANDIDATES = (
    Path("/content/drive/MyDrive/VQ_WallWork_CBM/source/PWDB_3275625"),
    Path("/content/drive/Shareddrives/VQ_WallWork_CBM/source/PWDB_3275625"),
)
DRIVE_PWDB_DIR = next(
    (path for path in DRIVE_PWDB_CANDIDATES if path.is_dir()),
    DRIVE_PWDB_CANDIDATES[0],
)

OUTPUT_BASE.mkdir(parents=True, exist_ok=True)
LOCAL_SOURCE.mkdir(parents=True, exist_ok=True)
LOCAL_XDG.mkdir(parents=True, exist_ok=True)

os.environ["XDG_DATA_HOME"] = str(LOCAL_XDG / "data")
os.environ["XDG_CACHE_HOME"] = str(LOCAL_XDG / "cache")
os.environ["XDG_STATE_HOME"] = str(LOCAL_XDG / "state")

if LOCAL_REPO.exists():
    shutil.rmtree(LOCAL_REPO)

print("Cloning PR #20 qualification branch...", flush=True)
subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", QUALIFICATION_REF, REPO_URL, str(LOCAL_REPO)],
    check=True,
)

CODE_REVISION = subprocess.check_output(
    ["git", "-C", str(LOCAL_REPO), "rev-parse", "HEAD"],
    text=True,
).strip()

print("Installing qualification candidate...", flush=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(LOCAL_REPO)],
    check=True,
)

RUNNER = LOCAL_REPO / "tests/full_data/parameterized_cohort_colab_validation.py"
STAGER = LOCAL_REPO / "tests/full_data/parameterized_cohort_colab_stage.py"
OUTPUT_ROOT = OUTPUT_BASE / CODE_REVISION[:12]
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("Setup: PASS")
print("Code revision:", CODE_REVISION)
print("Exact Drive PWDB directory:", DRIVE_PWDB_DIR)
print("Drive directory exists:", DRIVE_PWDB_DIR.is_dir())
print("Revision-scoped output root:", OUTPUT_ROOT)


## 2 — Stage the three required PWDB artifacts to local SSD

This cell does **not** walk `MyDrive`.

For each exact required file it:

- reuses a valid local-SSD copy if present;
- otherwise checks the exact configured `PWDB_3275625` directory;
- copies a Drive file once and checksums it locally;
- if missing or invalid, acquires only that artifact through VascuQuest's canonical verified source.

The cell prints progress for each artifact.


In [ ]:
stage_cmd = [
    sys.executable, str(STAGER),
    "--drive-source-dir", str(DRIVE_PWDB_DIR),
    "--local-source", str(LOCAL_SOURCE),
    "--report", str(OUTPUT_ROOT / "source_stage.json"),
]

print("Starting exact-file PWDB staging...", flush=True)
completed = subprocess.run(stage_cmd)
if completed.returncode != 0:
    raise RuntimeError(
        f"PWDB source staging failed with exit code {completed.returncode}"
    )

source_stage = json.loads((OUTPUT_ROOT / "source_stage.json").read_text())
print("\nPWDB local-SSD source gate: PASS")
for item in source_stage:
    print(
        f"  {item['artifact_id']}: {item['filename']} "
        f"({item['source_kind']}, {item['bytes']:,} bytes)"
    )


## 3 — Phase A: rapid smoke qualification

Runs **1 subject for each of the four disease conditions**. This must pass before Phase B.

The native cohort generator prints the subject being solved. Every completed subject is persisted to Drive and verified immediately.


In [ ]:
def run_qualification_phase(phase: str):
    result_name = "smoke_results.json" if phase == "smoke" else "full_results.json"
    cmd = [
        sys.executable, str(RUNNER),
        "--phase", phase,
        # Staging is already complete. Using LOCAL_SOURCE for both prevents
        # any further Google Drive scanning or source download.
        "--drive-pwdb-root", str(LOCAL_SOURCE),
        "--local-source", str(LOCAL_SOURCE),
        "--output-root", str(OUTPUT_ROOT),
        "--code-revision", CODE_REVISION,
    ]
    completed = subprocess.run(cmd)
    result_path = OUTPUT_ROOT / result_name
    if completed.returncode != 0:
        if result_path.exists():
            print("\nPersisted failure record:")
            print(result_path.read_text())
        raise RuntimeError(
            f"{phase} qualification failed with exit code {completed.returncode}; "
            f"see output above and {result_path}"
        )
    payload = json.loads(result_path.read_text())
    print(json.dumps(
        {"status": payload["status"], "cases_completed": payload["cases_completed"]},
        indent=2,
    ))
    return payload

smoke = run_qualification_phase("smoke")


In [ ]:
smoke = json.loads((OUTPUT_ROOT / "smoke_results.json").read_text())
if smoke.get("status") != "PASS" or smoke.get("cases_completed") != 4:
    raise RuntimeError("Smoke qualification is not complete/PASS.")
print("Smoke gate: PASS — full qualification authorized.")


## 4 — Phase B: 3 subjects × 4 diseases

Run this only after the previous cell reports `Smoke gate: PASS`.

Completed subjects are checkpointed to Drive. Rerun this cell after an interruption to resume.


In [ ]:
full = run_qualification_phase("full")

## 5 — Finalize machine-readable qualification evidence

In [ ]:
cmd = [
    sys.executable, str(RUNNER),
    "--phase", "finalize",
    "--drive-pwdb-root", str(LOCAL_SOURCE),
    "--local-source", str(LOCAL_SOURCE),
    "--output-root", str(OUTPUT_ROOT),
    "--code-revision", CODE_REVISION,
]
completed = subprocess.run(cmd)
if completed.returncode != 0:
    raise RuntimeError(f"finalize failed with exit code {completed.returncode}")

report_path = OUTPUT_ROOT / "parameterized-cohort-qualification.json"
report = json.loads(report_path.read_text())
print(json.dumps({
    "status": report["status"],
    "code_revision": report["code_revision"],
    "report": str(report_path),
    "scientific_boundary": report["scientific_boundary"],
}, indent=2))


After the final cell prints `PASS`, use the revision-scoped
`parameterized-cohort-qualification.json` as the evidence for deciding whether PR #20 is ready to merge.
